- feature extraction and feature engineering: transformation of raw data into features suitable for modeling;

- feature transformation: transformation of data to improve the accuracy of the algorithm;

- feature selection: removing unnecessary features.

###  predict the popularity of a new rental listing, i.e., classify the listing into three classes: ['low', 'medium' , 'high'].

In [4]:
# preload the dataset automatically if not in place

import os
from pathlib import Path
from pprint import pprint
import numpy as np
import pandas as pd

def check_file_exists(filename, outpath: Path, overwrite=False):
    file_exists = os.path.exists(f'{outpath}/{filename}')

    if(file_exists):
        print("file found in location")
    else:
        print("file missing")

In [5]:
FILE_NAME = "renthop_train.json.gz"
DATA_PATH = Path("./")

check_file_exists(filename=FILE_NAME, outpath=DATA_PATH)

file found in location


In [6]:
df = pd.read_json(DATA_PATH / FILE_NAME, compression="gzip", convert_dates=["created"])
df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


**bag of words** - we create a vector with the length of the vocabulary, compute the number of occurrences of each word in the text, and place that number of occurrences in the appropriate position in the vector.

In [7]:
# enumerate adds indexes

fruits = ["apple", "banana", "orange"]

result = list(enumerate(fruits))
print(result)

[(0, 'apple'), (1, 'banana'), (2, 'orange')]


In [13]:
texts = ["i have a cat", "you have a dog", "you and i have a cat and a dog"]

vocabulary = list(
    enumerate(set([word for sentence in texts for word in sentence.split()]))
)

print("vocabulary", vocabulary)

def vectorizer(text):
    vector = np.zeros(len(vocabulary))
    for i, word in vocabulary:
        num = 0
        for w in text:
            if w == word:
                num += 1
        if num:
            vector[i] = num
    return vector

print("Vectors:")
for sentence in texts:
    print(vectorizer(sentence.split()))

vocabulary [(0, 'have'), (1, 'dog'), (2, 'i'), (3, 'and'), (4, 'you'), (5, 'a'), (6, 'cat')]
Vectors:
[1. 0. 1. 0. 0. 1. 1.]
[1. 1. 0. 0. 1. 1. 0.]
[1. 1. 1. 2. 1. 2. 1.]


When using algorithms like Bag of Words, we lose the order of the words in the text, which means that the texts “i have no cows” and “no, i have cows” will appear identical after vectorization when, in fact, they have the opposite meaning. To avoid this problem, we can revisit our tokenization step and use N-grams (the sequence of N consecutive tokens) instead.

Great question. The key thing you're missing is **what `CountVectorizer` considers to be the vocabulary**.

Let's go through it step by step.

### Step 1: The documents

```python
doc1 = "no i have cows"
doc2 = "i have no cows"
```

---

### Step 2: Build the vocabulary

Since `ngram_range=(1,1)`, we only look at **individual words**.

Across both documents, the unique words are:

```text
no
i
have
cows
```

`CountVectorizer` sorts them alphabetically by default:

```python
['cows', 'have', 'no']
```

Wait...where did **"i"** go?

### Step 3: Why is "i" missing?

By default, `CountVectorizer` uses the regular expression

```python
token_pattern=r"(?u)\b\w\w+\b"
```

This means:

> Only keep tokens that are **2 or more characters long**.

Since `"i"` is only **one character**, it is **ignored**.

So the vocabulary becomes:

| Index | Word |
| ----: | ---- |
|     0 | cows |
|     1 | have |
|     2 | no   |

You can verify this:

```python
vect = CountVectorizer()

vect.fit(["no i have cows", "i have no cows"])

print(vect.vocabulary_)
```

Output:

```python
{'cows': 0, 'have': 1, 'no': 2}
```

---

### Step 4: Count the words

Document 1:

```text
no i have cows
```

Ignoring `"i"`:

```text
no
have
cows
```

Counts:

| Word | Count |
| ---- | ----: |
| cows |     1 |
| have |     1 |
| no   |     1 |

Vector:

```python
[1, 1, 1]
```

---

Document 2:

```text
i have no cows
```

Ignoring `"i"`:

```text
have
no
cows
```

Counts:

| Word | Count |
| ---- | ----: |
| cows |     1 |
| have |     1 |
| no   |     1 |

Vector:

```python
[1, 1, 1]
```

---

### Final matrix

```python
array([
    [1, 1, 1],
    [1, 1, 1]
])
```

Each **row** is a document.

Each **column** is a vocabulary word.

| Document       | cows | have | no |
| -------------- | ---: | ---: | -: |
| no i have cows |    1 |    1 |  1 |
| i have no cows |    1 |    1 |  1 |

---

## If you want to keep `"i"`

Tell `CountVectorizer` to accept single-letter words:

```python
vect = CountVectorizer(token_pattern=r"(?u)\b\w+\b")

X = vect.fit_transform(["no i have cows", "i have no cows"])

print(vect.get_feature_names_out())
print(X.toarray())
```

Output:

```python
['cows' 'have' 'i' 'no']

array([
    [1, 1, 1, 1],
    [1, 1, 1, 1]
])
```

Now `"i"` appears because you've changed the token pattern to allow words of length **1 or more**.


In [17]:
from sklearn.feature_extraction.text import CountVectorizer
vect = CountVectorizer(ngram_range=(1,1))
vect.fit_transform(["no i have cows", "i have no cows"]).toarray()

array([[1, 1, 1],
       [1, 1, 1]])

In [18]:
vect.vocabulary_

{'no': 2, 'have': 1, 'cows': 0}

In [19]:
vect = CountVectorizer(ngram_range=(1,2))
vect.fit_transform(["no i have cows", "i have no cows"]).toarray()

array([[1, 1, 1, 0, 1, 0, 1],
       [1, 1, 0, 1, 1, 1, 0]])

In [20]:
vect.vocabulary_

{'no': 4,
 'have': 1,
 'cows': 0,
 'no have': 6,
 'have cows': 2,
 'have no': 3,
 'no cows': 5}

# How the vocabulary is connected to the array

The connection is that **the vocabulary assigns each token/ngram to a column index** in the output matrix.

Let's go through it step by step.

### Step 1. The documents

```python
doc1 = "no i have cows"
doc2 = "i have no cows"
```

Since `CountVectorizer` ignores **single-character words** by default (`token_pattern=r"(?u)\b\w\w+\b"`), the word `"i"` is discarded.

So the documents become:

```
doc1: no have cows
doc2: have no cows
```

---

### Step 2. Generate unigrams and bigrams

With

```python
ngram_range=(1,2)
```

we generate both 1-word and 2-word sequences.

#### Document 1

Tokens:

```
no
have
cows
```

Unigrams:

```
no
have
cows
```

Bigrams:

```
no have
have cows
```

---

#### Document 2

Tokens:

```
have
no
cows
```

Unigrams:

```
have
no
cows
```

Bigrams:

```
have no
no cows
```

---

### Step 3. Build the vocabulary

The vocabulary is

```python
{
 'cows': 0,
 'have': 1,
 'have cows': 2,
 'have no': 3,
 'no': 4,
 'no cows': 5,
 'no have': 6
}
```

Each value is the **column number** in the matrix.

| Column | Feature   |
| -----: | --------- |
|      0 | cows      |
|      1 | have      |
|      2 | have cows |
|      3 | have no   |
|      4 | no        |
|      5 | no cows   |
|      6 | no have   |

---

### Step 4. Fill the matrix

#### First document

Contains

* ✅ cows
* ✅ have
* ✅ have cows
* ❌ have no
* ✅ no
* ❌ no cows
* ✅ no have

So the counts become

| Feature   | Count |
| --------- | ----: |
| cows      |     1 |
| have      |     1 |
| have cows |     1 |
| have no   |     0 |
| no        |     1 |
| no cows   |     0 |
| no have   |     1 |

which gives

```python
[1, 1, 1, 0, 1, 0, 1]
```

---

#### Second document

Contains

* ✅ cows
* ✅ have
* ❌ have cows
* ✅ have no
* ✅ no
* ✅ no cows
* ❌ no have

So

```python
[1, 1, 0, 1, 1, 1, 0]
```

---

## Visualizing the whole matrix

| Document | cows | have | have cows | have no | no | no cows | no have |
| -------- | ---: | ---: | --------: | ------: | -: | ------: | ------: |
| doc1     |    1 |    1 |         1 |       0 |  1 |       0 |       1 |
| doc2     |    1 |    1 |         0 |       1 |  1 |       1 |       0 |

This is exactly

```python
array([
    [1, 1, 1, 0, 1, 0, 1],
    [1, 1, 0, 1, 1, 1, 0]
])
```

---

### You can verify the mapping in code

```python
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(1,2))

X = vect.fit_transform(["no i have cows", "i have no cows"])

print(vect.vocabulary_)
print(vect.get_feature_names_out())
print(X.toarray())
```

Output:

```python
['cows',
 'have',
 'have cows',
 'have no',
 'no',
 'no cows',
 'no have']

[[1 1 1 0 1 0 1]
 [1 1 0 1 1 1 0]]
```

`get_feature_names_out()` returns the features in the exact order of the columns in the matrix, making it much easier to interpret than reading the raw `vocabulary_` dictionary.


Also note that one does not have to use only words. In some cases, it is possible to generate N-grams of characters. This approach would be able to account for similarity of related words or handle typos.

In [3]:
from scipy.spatial.distance import euclidean
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(3,3), analyzer="char_wb")

n1, n2, n3, n4 = vect.fit_transform(
     ["andersen", "petersen", "petrov", "smith"]
).toarray()

euclidean(n1, n2), euclidean(n2, n3), euclidean(n3, n4)

(np.float64(2.8284271247461903),
 np.float64(3.1622776601683795),
 np.float64(3.3166247903554))

## Images

In [ ]:
## install tensorflow and keras - (currently not supported in python 3.14 - install on .11 or .12)
# from keras.applications 